# Notebook RMSE


This notebook computes the globally and temporally averaged Root Mean Square Error (RMSE) between reconstructed fields and the reference fields for all configurations, within the OSSE over the Hawaii region.

The resulting metrics are averaged over the whole domain and time window, and are used in the tables presented in the manuscript.

In [ ]:
import xarray as xr 
import matplotlib.pyplot as plt 
import numpy as np 
import pandas as pd

import xskillscore

from pyinterp import fill, Axis, TemporalAxis, Grid3D, Grid2D
n_workers = 10

from cartopy.mpl.ticker import LongitudeFormatter, LatitudeFormatter
import matplotlib.ticker as mticker 

In [ ]:
path_data = "../.." # Change path according to where data are saved

## Importing data

QG reconstruction 

In [ ]:
ds_QG = xr.open_mfdataset(f"{path_data}/data/mapping_outputs/config_QG/*.nc")
ssh_bm_QG = ds_QG.ssh_bm.load()
ssh_bm_QG = ssh_bm_QG.rename({"lon":"longitude","lat":"latitude"})

QG/SW reconstruction 

In [ ]:
ds_QGSW = xr.open_mfdataset(f"{path_data}/data/mapping_outputs/config_QGSW/*.nc")
# - BALANCED MOTIONS BM - #
ssh_bm_QGSW = ds_QGSW.ssh_bm.load()
ssh_bm_QGSW = ssh_bm_QGSW.rename({"lon":"longitude","lat":"latitude"})
# - INTERNAL TIDE IT - #
ssh_it_QGSW = ds_QGSW.ssh_it.load()
ssh_it_QGSW = ssh_it_QGSW.rename({"lon":"longitude","lat":"latitude"})

QG/SW reconstruction with no time control for IT

In [ ]:
ds_QGSW_notime = xr.open_mfdataset(f"{path_data}/data/mapping_outputs/config_QGSW_notime/*.nc")
# - BALANCED MOTIONS BM - #
ssh_bm_QGSW_notime = ds_QGSW_notime.ssh_bm.load()
ssh_bm_QGSW_notime = ssh_bm_QGSW_notime.rename({"lon":"longitude","lat":"latitude"})
# - INTERNAL TIDE IT - #
ssh_it_QGSW_notime = ds_QGSW_notime.ssh_it.load()
ssh_it_QGSW_notime = ssh_it_QGSW_notime.rename({"lon":"longitude","lat":"latitude"})

Reference dataset 

In [ ]:
%run ./functions_interp.ipynb

In [ ]:
ds_truth = xr.open_mfdataset(f"{path_data}/data/OSSE/ref/*.nc")
ds_truth = ds_truth.sel(longitude=slice(ssh_bm_QG.longitude[0],ssh_bm_QG.longitude[-1]),
                        latitude=slice(ssh_bm_QG.latitude[0],ssh_bm_QG.latitude[-1]),
                        time = slice(ssh_bm_QG.time[0],ssh_bm_QG.time[-1]),drop=True)
ssh_bm_truth = ds_truth.ssh_bm.load()
ssh_it_12h_truth = ds_truth.ssh_it_12h.load()
ssh_it_truth = ds_truth.ssh_it.load()
ssh_tot_truth = ds_truth.ssh.load()

In [ ]:
ssh_bm_truth = fill_nan(ssh_bm_truth)
ssh_it_truth = fill_nan(ssh_it_truth)
ssh_tot_truth = fill_nan(ssh_tot_truth)

## Common parameters 

Starting timestep and ending timestep for computing RMSE 

In [ ]:
i_start = 10*24 # Starting timestep, in hours
i_end = -5*24 # Ending timestep, in hours

## Computing statistical analysis

### Whole domain 

Values of RMSE 

In [ ]:
## - SSH BM - ##
rmse_bm_QG = xskillscore.rmse(ssh_bm_QG[i_start:i_end,:,:], ssh_bm_truth[i_start:i_end,:,:], dim=['longitude', 'latitude','time'], skipna=True).compute()
rmse_bm_QGSW = xskillscore.rmse(ssh_bm_QGSW[i_start:i_end,:,:], ssh_bm_truth[i_start:i_end,:,:], dim=['longitude', 'latitude','time'], skipna=True).compute()
rmse_bm_QGSW_notime = xskillscore.rmse(ssh_bm_QGSW_notime[i_start:i_end,:,:], ssh_bm_truth[i_start:i_end,:,:], dim=['longitude', 'latitude','time'], skipna=True).compute()
## - SSH IT - ## 
rmse_it_QGSW = xskillscore.rmse(ssh_it_QGSW[i_start:i_end,:,:], ssh_it_truth[i_start:i_end,:,:], dim=['longitude', 'latitude','time'], skipna=True).compute()
rmse_it_QGSW_notime = xskillscore.rmse(ssh_it_QGSW_notime[i_start:i_end,:,:], ssh_it_truth[i_start:i_end,:,:], dim=['longitude', 'latitude','time'], skipna=True).compute()

In [ ]:
print("SSH BM")
print("rmse_bm_QG : ",f"{rmse_bm_QG.values*100:.3g} cm")
print("rmse_bm_QGSW : ",f"{rmse_bm_QGSW.values*100:.3g} cm")
print("rmse_bm_QGSW_notime : ",f"{rmse_bm_QGSW_notime.values*100:.3g} cm")
print("SSH IT")
print("rmse_it_QGSW : ",f"{rmse_it_QGSW.values*100:.3g} cm")
print("rmse_it_QGSW_notime : ",f"{rmse_it_QGSW_notime.values*100:.3g} cm")

Values of $\sigma_{SCORE}$ 

In [ ]:
## - SSH BM - ##
score_bm_QG = 1-(rmse_bm_QG**2/np.var(ssh_bm_truth[i_start:i_end,:,:]))
score_bm_QGSW = 1-(rmse_bm_QGSW**2/np.var(ssh_bm_truth[i_start:i_end,:,:]))
score_bm_QGSW_notime = 1-(rmse_bm_QGSW_notime**2/np.var(ssh_bm_truth[i_start:i_end,:,:]))
## - SSH IT - ## 
score_it_QGSW = 1-(rmse_it_QGSW**2/np.var(ssh_it_truth[i_start:i_end,:,:]))
score_it_QGSW_notime = 1-(rmse_it_QGSW_notime**2/np.var(ssh_it_truth[i_start:i_end,:,:]))

In [ ]:
print("SSH BM")
print("score_bm_QG : ",f"{100*score_bm_QG.values:.3g} %")
print("score_bm_QGSW : ",f"{100*score_bm_QGSW.values:.3g} %")
print("score_bm_QGSW_notime : ",f"{100*score_bm_QGSW_notime.values:.3g} %")
print("SSH IT")
print("score_it_QGSW : ",f"{100*score_it_QGSW.values:.3g} %")
print("score_it_QGSW_notime : ",f"{100*score_it_QGSW_notime.values:.3g} %")

### Under SWOT swaths

In [ ]:
mask_swot = np.load("../mapping/aux/mask_swot.npy")

Values of RMSE inside swath 

In [ ]:
## - SSH BM - ##
rmse_bm_QG_swath = xskillscore.rmse(ssh_bm_QG[i_start:,:,:].where(mask_swot), ssh_bm_truth[i_start:,:,:].where(mask_swot), dim=['longitude', 'latitude','time'], skipna=True).compute()
rmse_bm_QGSW_swath = xskillscore.rmse(ssh_bm_QGSW[i_start:,:,:].where(mask_swot), ssh_bm_truth[i_start:,:,:].where(mask_swot), dim=['longitude', 'latitude','time'], skipna=True).compute()
rmse_bm_QGSW_notime_swath = xskillscore.rmse(ssh_bm_QGSW_notime[i_start:,:,:].where(mask_swot), ssh_bm_truth[i_start:,:,:].where(mask_swot), dim=['longitude', 'latitude','time'], skipna=True).compute()
# ## - SSH IT - ## 
rmse_it_QGSW_swath = xskillscore.rmse(ssh_it_QGSW[i_start:,:,:].where(mask_swot), ssh_it_truth[i_start:,:,:].where(mask_swot), dim=['longitude', 'latitude','time'], skipna=True).compute()
rmse_it_QGSW_notime_swath = xskillscore.rmse(ssh_it_QGSW_notime[i_start:,:,:].where(mask_swot), ssh_it_truth[i_start:,:,:].where(mask_swot), dim=['longitude', 'latitude','time'], skipna=True).compute()

In [ ]:
print("SSH BM inside swath")
print("rmse_bm_QG : ",f"{100*rmse_bm_QG_swath.values:.3g} cm")
print("rmse_bm_QGSW : ",f"{100*rmse_bm_QGSW_swath.values:.3g} cm")
print("rmse_bm_notime_QGSW : ",f"{100*rmse_bm_QGSW_notime_swath.values:.3g} cm")
print("SSH IT inside swath")
print("rmse_it_QGSW : ",f"{100*rmse_it_QGSW_swath.values:.3g} cm")
print("rmse_it_QGSW_notime : ",f"{100*rmse_it_QGSW_notime_swath.values:.3g} cm")


Values of $\sigma_{SCORE}$ inside swath 

In [ ]:
## - SSH BM - ##
score_bm_QG_swath = 1-(rmse_bm_QG_swath**2/np.var(ssh_bm_truth[i_start:i_end,:,:].where(mask_swot)))
score_bm_QGSW_swath = 1-(rmse_bm_QGSW_swath**2/np.var(ssh_bm_truth[i_start:i_end,:,:].where(mask_swot)))
score_bm_QGSW_notime_swath = 1-(rmse_bm_QGSW_notime_swath**2/np.var(ssh_bm_truth[i_start:,:,:].where(mask_swot)))
## - SSH IT - ## 
score_it_QGSW_swath = 1-(rmse_it_QGSW_swath**2/np.var(ssh_it_truth[i_start:i_end,:,:].where(mask_swot)))
score_it_QGSW_notime_swath = 1-(rmse_it_QGSW_notime_swath**2/np.var(ssh_it_truth[i_start:,:,:].where(mask_swot)))

In [ ]:
print("SSH BM inside swath")
print("score_bm_QG : ",f"{100*score_bm_QG_swath.values:.3g} %")
print("score_bm_QGSW : ",f"{100*score_bm_QGSW_swath.values:.3g} %")
print("score_bm_QGSW_notime : ",f"{100*score_bm_QGSW_notime_swath.values:.3g} %")
print("SSH IT inside swath")
print("score_it_QGSW : ",f"{100*score_it_QGSW_swath.values:.3g} %")
print("score_it_QGSW_notime : ",f"{100*score_it_QGSW_notime_swath.values:.3g} %")